# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hapepaAhmed/my-capstone-project/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from google.colab import userdata
from huggingface_hub import hf_hub_download

In [2]:
HF_TOKEN = userdata.get("HF_TOKEN")

In [3]:
parquet_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN,
)

In [4]:
df = pd.read_parquet(parquet_path)

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [5]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

print("Model:", model.__class__.__name__)


Model: XGBRegressor


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [6]:
import pandas as pd
import numpy as np

# Create modeling dataframe
model_df = df.copy()

# CTR
model_df["ctr"] = np.where(
    model_df["gsc_impressions"] > 0,
    model_df["gsc_clicks"] / model_df["gsc_impressions"],
    0
)

# Engagement rate
model_df["engagement_rate"] = np.where(
    model_df["ga4_sessions"] > 0,
    model_df["ga4_engaged_sessions"] /
    model_df["ga4_sessions"],
    0
)

# Average engagement time per session
model_df["avg_engagement_sec"] = np.where(
    model_df["ga4_sessions"] > 0,
    model_df["ga4_total_engagement_sec"] /
    model_df["ga4_sessions"],
    0
)

# Position bucket
model_df["position_bucket"] = pd.cut(
    model_df["gsc_avg_position"],
    bins=[0, 10, 20, 50, float("inf")],
    labels=["Top10", "11-20", "21-50", "50+"]
)

# Convert position bucket to numeric codes
model_df["position_bucket"] = (
    model_df["position_bucket"].cat.codes
)

print("Modeling dataframe shape:", model_df.shape)

Modeling dataframe shape: (9841378, 34)


In [7]:
print(
    model_df[
        [
            "ctr",
            "engagement_rate",
            "avg_engagement_sec",
            "position_bucket"
        ]
    ].head()
)

     ctr  engagement_rate  avg_engagement_sec  position_bucket
0  0.000              0.0                 0.0                0
1  0.000              0.0                 0.0               -1
2  0.008              0.0                 0.0                0
3  0.000              0.0                 0.0                0
4  0.000              0.0                 0.0                0


In [8]:
selected_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ctr",
    "engagement_rate",
    "avg_engagement_sec",
    "position_bucket"
]

print("Selected features:")
for feature in selected_features:
    print("-", feature)

Selected features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ctr
- engagement_rate
- avg_engagement_sec
- position_bucket


In [1]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_df,
        groups=model_df["client_hash_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print(
    "Train clients:",
    train_df["client_hash_id"].nunique()
)

print(
    "Test clients:",
    test_df["client_hash_id"].nunique()
)

overlap = (
    set(train_df["client_hash_id"])
    & set(test_df["client_hash_id"])
)

print("Client overlap:", len(overlap))

NameError: name 'model_df' is not defined

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
X_train = train_df[selected_features].copy()
X_test = test_df[selected_features].copy()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.